In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

spark = SparkSession.builder.getOrCreate()

customer_data = [
    (1, "Akash", "Mumbai", "2023-01-10"),
    (2, "Rahul", "Pune", "2023-02-15"),
    (3, "Sneha", "Delhi", "2023-03-20"),
    (4, "Priya", "Bangalore", "2023-04-25"),
    (5, "Amit", "Hyderabad", "2023-05-30"),
    (6, "Neha", "Chennai", "2023-06-12")
]

customer_schema = StructType([
    StructField("customer_id", IntegerType(), True),
    StructField("customer_name", StringType(), True),
    StructField("city", StringType(), True),
    StructField("signup_date", StringType(), True)
])

customer_df = spark.createDataFrame(customer_data, customer_schema)

customer_df = customer_df.withColumn('signup_date',to_date(col('signup_date')))

order_data = [
    (101, 1, "2024-09-01", 2500, "Completed"),
    (102, 1, "2024-10-15", 1800, "Completed"),
    (103, 2, "2024-08-10", 1200, "Cancelled"),
    (104, 2, "2024-09-20", 2200, "Completed"),
    (105, 3, "2024-06-05", 3000, "Completed"),
    (106, 3, "2024-07-18", 1500, "Returned"),
    (107, 4, "2024-11-01", 4000, "Completed"),
    (108, 5, "2024-03-22", 1700, "Completed"),
    (109, 5, "2024-04-30", 2600, "Completed")
]

order_schema = StructType([
    StructField("order_id", IntegerType(), True),
    StructField("customer_id", IntegerType(), True),
    StructField("order_date", StringType(), True),
    StructField("order_amount", IntegerType(), True),
    StructField("order_status", StringType(), True)
])

orders_df = spark.createDataFrame(order_data, order_schema)
orders_df = orders_df.withColumn('order_date',to_date(col('order_date')))


In [ ]:
# Find the total number of Completed orders placed by each customer.
tota_ord =orders_df.filter(col('order_status')=='Completed')\
         .groupBy('customer_id').agg(count('order_id').alias('total_orders'))

final_df = customer_df.join(
    tota_ord,
    on="customer_id",
    how="left"
)

final_df.select("customer_id", "customer_name", "total_orders").show()

In [21]:
# Find customers who have not placed any orders at all.
tota_ord =orders_df.groupBy('customer_id').agg(count('order_id').alias('total_orders'))

final_df = customer_df.join(
    tota_ord,
    on="customer_id",
    how="left"
)

final_df.filter(col('total_orders').isNull()).select("customer_id", "customer_name", "total_orders").show()

+-----------+-------------+------------+
|customer_id|customer_name|total_orders|
+-----------+-------------+------------+
|          6|         Neha|        NULL|
+-----------+-------------+------------+



In [ ]:
#Find the total number of orders placed by each customer.
orders_df.groupBy('customer_id').agg(count('order_id').alias('total_orders')).show()

In [ ]:
# Find customers who have placed MORE THAN 1 order.
orders_df.groupBy('customer_id').agg(count('order_id').alias('total_orders'))\
         .filter(col('total_orders')>1)\
         .show()

In [ ]:
# get last order date per customer
last_order_df = orders_df.groupBy("customer_id") \
    .agg(max("order_date").alias("last_order_date"))

# filter customers whose last order is older than 3 months
inactive_customers = last_order_df.filter(
    col("last_order_date") < date_sub(current_date(), 3)
)
inactive_customers.select("customer_id").show()
# last_order_df.show()


In [ ]:
from pyspark.sql.functions import sum

total_revenue_df = orders_df.groupBy("customer_id") \
    .agg(sum("order_amount").alias("total_revenue"))

final_df = customer_df.join(
    total_revenue_df,
    on="customer_id",
    how="left"
)

final_df.select("customer_id", "customer_name", "total_revenue").show()

